# Final Project Worked Example  
## Conceptual Change in a Small Historical Corpus: Economy, Culture, and Modernization

**Course:** Python for Digital Humanities  
**Format:** Jupyter Notebook / Google Colab  
**Project type:** Text analysis  
**Difficulty level:** Baseline-to-intermediate final project  

---

## Research Question

How does the vocabulary of **modernization** change across a small historical corpus of fictionalized institutional reports from 1936–1940?

More specifically:

1. Which concepts occur most frequently?
2. Do terms related to **economy**, **culture**, and **technology** change over time?
3. Which words tend to occur together in the same documents?
4. What can these patterns tell us about the discourse of institutional modernization?

---

## Important Note About the Dataset

This notebook uses a **small synthetic teaching dataset** created inside the notebook.  
It is not a real historical archive. It imitates the structure of a Digital Humanities text corpus so that the full workflow is reproducible without external files.

For a real final project, students should replace this synthetic dataset with a real corpus, for example:

- digitized newspapers
- institutional reports
- archival metadata
- museum descriptions
- parliamentary debates
- OCR text collections

# 1. Setup

We use standard Python libraries:

- `pandas` for tabular data
- `re` for regular expressions
- `collections.Counter` for word counting
- `matplotlib` for plots
- `networkx` for a simple co-occurrence network

In [ ]:
import re
from collections import Counter
from itertools import combinations

import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    !pip install networkx
    import networkx as nx

pd.set_option("display.max_colwidth", 120)

# 2. Create a Small Teaching Corpus

Each row represents one document.

The columns are:

- `doc_id`: unique document identifier
- `year`: year of publication
- `genre`: simplified document type
- `title`: document title
- `text`: document body

In a real project, this data would usually be loaded from `.txt`, `.csv`, `.json`, XML, or an archive export.

In [ ]:
documents = [
    {
        "doc_id": "D1936_01",
        "year": 1936,
        "genre": "economic report",
        "title": "Trade and Rural Production",
        "text": """
        The ministry reviewed trade, agriculture, exports, and rural production.
        Farmers reported stable grain prices, while timber exports improved.
        Roads and railway connections were described as necessary for modern commerce.
        """
    },
    {
        "doc_id": "D1936_02",
        "year": 1936,
        "genre": "cultural report",
        "title": "Libraries and Public Education",
        "text": """
        Public libraries expanded reading rooms and education programmes.
        Cultural societies promoted lectures, books, language study, and national heritage.
        The report connected literacy with civic development and social improvement.
        """
    },
    {
        "doc_id": "D1936_03",
        "year": 1936,
        "genre": "administrative report",
        "title": "Regional Administration and Infrastructure",
        "text": """
        Regional offices discussed roads, postal service, administration, and village communication.
        Infrastructure was seen as a condition for economic coordination and public order.
        """
    },
    {
        "doc_id": "D1937_01",
        "year": 1937,
        "genre": "economic report",
        "title": "Industry and Export Markets",
        "text": """
        Industry increased production in textiles, timber, and food processing.
        Export markets required better transport, accounting, and technical training.
        Economic policy emphasized efficiency, productivity, and cooperation.
        """
    },
    {
        "doc_id": "D1937_02",
        "year": 1937,
        "genre": "cultural report",
        "title": "Museums, Schools, and National Memory",
        "text": """
        Museums and schools cooperated to preserve memory, heritage, and language.
        Teachers organized exhibitions and reading circles.
        Culture was described as a foundation for national education and identity.
        """
    },
    {
        "doc_id": "D1937_03",
        "year": 1937,
        "genre": "technical report",
        "title": "Radio, Communication, and Public Information",
        "text": """
        Radio communication developed rapidly in towns and rural districts.
        Technical specialists discussed electricity, broadcasting, and public information.
        Modern communication was praised for connecting citizens with state institutions.
        """
    },
    {
        "doc_id": "D1938_01",
        "year": 1938,
        "genre": "economic report",
        "title": "Finance, Credit, and Modern Accounting",
        "text": """
        Banks expanded credit for agriculture, trade, and small industry.
        Modern accounting methods were introduced in cooperatives.
        Finance was presented as a tool for rational planning and economic stability.
        """
    },
    {
        "doc_id": "D1938_02",
        "year": 1938,
        "genre": "cultural report",
        "title": "Theatre, Literature, and Public Culture",
        "text": """
        Theatre groups, writers, and libraries supported public culture.
        Literature was discussed together with language, education, and civic morality.
        Cultural development depended on access to books, lectures, and local institutions.
        """
    },
    {
        "doc_id": "D1938_03",
        "year": 1938,
        "genre": "technical report",
        "title": "Mechanization and Agricultural Technology",
        "text": """
        Mechanization entered agricultural production through tractors, machines, and improved tools.
        Technical education was needed for maintenance and efficient use.
        Technology was linked to productivity, planning, and rural modernization.
        """
    },
    {
        "doc_id": "D1939_01",
        "year": 1939,
        "genre": "economic report",
        "title": "Trade Disruption and Strategic Supplies",
        "text": """
        International tension affected trade, imports, exports, and strategic supplies.
        Officials discussed reserves, transport security, and industrial coordination.
        Economic planning became more urgent in response to uncertainty.
        """
    },
    {
        "doc_id": "D1939_02",
        "year": 1939,
        "genre": "cultural report",
        "title": "Education During Uncertainty",
        "text": """
        Schools and libraries continued education despite political uncertainty.
        Cultural institutions emphasized unity, language, memory, and discipline.
        Reading programmes were connected with civic responsibility.
        """
    },
    {
        "doc_id": "D1939_03",
        "year": 1939,
        "genre": "technical report",
        "title": "Transport, Radio, and Emergency Coordination",
        "text": """
        Transport networks, radio stations, and administrative offices prepared emergency coordination.
        Communication technology became important for security, planning, and state capacity.
        """
    },
    {
        "doc_id": "D1940_01",
        "year": 1940,
        "genre": "economic report",
        "title": "Planning and Institutional Reorganization",
        "text": """
        Economic offices reported reorganization, planning, production quotas, and administrative control.
        Trade, finance, and industry were increasingly described through coordination and regulation.
        """
    },
    {
        "doc_id": "D1940_02",
        "year": 1940,
        "genre": "cultural report",
        "title": "Libraries and Ideological Education",
        "text": """
        Libraries, schools, and cultural committees revised education programmes.
        Language, literature, and public lectures were connected with ideology and collective discipline.
        """
    },
    {
        "doc_id": "D1940_03",
        "year": 1940,
        "genre": "technical report",
        "title": "Technology and Central Planning",
        "text": """
        Technology was described as essential for central planning, transport, communication, and production.
        Engineers emphasized machines, electricity, radio, and industrial discipline.
        """
    }
]

df = pd.DataFrame(documents)
df

# 3. Dataset Overview

Before analysis, inspect the structure of the dataset.

A good final project should always describe:

- number of documents
- time span
- available metadata
- possible limitations

In [ ]:
print("Number of documents:", len(df))
print("Years:", df["year"].min(), "-", df["year"].max())
print("Genres:", ", ".join(sorted(df["genre"].unique())))

df.groupby(["year", "genre"]).size().unstack(fill_value=0)

In [ ]:
genre_counts = df["genre"].value_counts()

genre_counts.plot(kind="bar")
plt.title("Number of Documents by Genre")
plt.xlabel("Genre")
plt.ylabel("Document Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Initial Observation

The teaching corpus is deliberately balanced: each year contains economic, cultural, and technical/administrative material.  
This makes comparisons easier, but it is also artificial. Real archives are rarely this clean.

# 4. Text Preprocessing

We will:

1. lowercase text
2. extract alphabetic tokens
3. remove common stopwords
4. store token lists for each document

For real multilingual projects, stopword handling should be adapted to the corpus language.

In [ ]:
STOPWORDS = {
    "the", "and", "for", "was", "were", "with", "as", "a", "an", "in", "of", "to",
    "on", "through", "by", "from", "into", "at", "is", "are", "be", "became",
    "that", "this", "it", "its", "their", "while", "better",
    "more", "small", "local"
}

def tokenize(text):
    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    return [token for token in tokens if token not in STOPWORDS and len(token) > 2]

df["tokens"] = df["text"].apply(tokenize)
df["token_count"] = df["tokens"].apply(len)

df[["doc_id", "year", "genre", "title", "token_count", "tokens"]].head()

# 5. Analysis 1: Most Frequent Words

This is a baseline method. It is simple, but useful for orientation.

Frequency analysis alone is not enough for a strong DH project, but it helps identify major vocabulary clusters.

In [ ]:
all_tokens = [token for tokens in df["tokens"] for token in tokens]
word_counts = Counter(all_tokens)

top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "count"])
top_words

In [ ]:
top_words.sort_values("count").plot(
    kind="barh",
    x="word",
    y="count",
    legend=False
)
plt.title("Top 20 Words in the Corpus")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.tight_layout()
plt.show()

## Interpretation of Frequency Results

The most frequent words already suggest the main thematic fields of the corpus:

- institutional modernization: `planning`, `coordination`, `administration`
- economy: `trade`, `industry`, `production`, `finance`
- culture and education: `libraries`, `education`, `language`, `culture`
- technology and infrastructure: `radio`, `communication`, `transport`, `technology`

This supports the idea that modernization is not only economic. It appears across cultural, technical, and administrative domains.

# 6. Analysis 2: Concept Groups Over Time

Instead of counting individual words only, we define interpretive concept groups.

This is a common DH move: the researcher makes explicit categories and then tests how they appear in the corpus.

Our concept groups:

- **Economy:** trade, industry, finance, exports, credit, production
- **Culture:** libraries, education, language, literature, museums, theatre
- **Technology:** radio, communication, technology, machines, electricity, transport
- **Administration:** planning, coordination, administration, regulation, control

In [ ]:
concept_groups = {
    "economy": {"trade", "industry", "finance", "exports", "credit", "production", "commerce", "banks"},
    "culture": {"libraries", "education", "language", "literature", "museums", "theatre", "heritage", "culture"},
    "technology": {"radio", "communication", "technology", "machines", "electricity", "transport", "mechanization", "technical"},
    "administration": {"planning", "coordination", "administration", "regulation", "control", "offices", "institutions", "state"}
}

def count_concepts(tokens, vocabulary):
    return sum(1 for token in tokens if token in vocabulary)

for concept, vocab in concept_groups.items():
    df[concept] = df["tokens"].apply(lambda tokens: count_concepts(tokens, vocab))

df[["doc_id", "year", "genre", "economy", "culture", "technology", "administration"]]

In [ ]:
concept_by_year = df.groupby("year")[list(concept_groups.keys())].sum()
concept_by_year

In [ ]:
concept_by_year.plot(marker="o")
plt.title("Concept Group Frequencies by Year")
plt.xlabel("Year")
plt.ylabel("Frequency")
plt.xticks(concept_by_year.index)
plt.tight_layout()
plt.show()

## Interpretation of Concept Trends

In this small teaching corpus, the vocabulary of modernization becomes increasingly administrative and technical by 1939–1940.

The pattern suggests a shift from general cultural and economic development toward:

- planning
- coordination
- regulation
- emergency preparedness
- state capacity

A real historical interpretation would require external contextual knowledge, but the computational pattern gives us a strong starting point for further reading.

# 7. Analysis 3: Keyword-in-Context (KWIC)

KWIC shows how a word appears in context.

This is useful because word counts can be misleading.  
For example, `planning` may refer to economic planning, educational planning, or technical planning.

In [ ]:
def kwic(df, keyword, window=6):
    rows = []
    keyword = keyword.lower()

    for _, row in df.iterrows():
        tokens = tokenize(row["text"])
        for i, token in enumerate(tokens):
            if token == keyword:
                left = " ".join(tokens[max(0, i-window):i])
                right = " ".join(tokens[i+1:i+1+window])
                rows.append({
                    "doc_id": row["doc_id"],
                    "year": row["year"],
                    "title": row["title"],
                    "left_context": left,
                    "keyword": token,
                    "right_context": right
                })
    return pd.DataFrame(rows)

kwic(df, "planning", window=5)

## Interpretation of KWIC Results

The word `planning` appears in relation to finance, technology, production, emergency coordination, and central administration.  
This supports the interpretation that modernization is being framed as an increasingly coordinated institutional process.

In a real project, this section should include close reading of selected examples.

# 8. Analysis 4: Word Co-occurrence Network

A co-occurrence network connects words that appear in the same document.

This is a simplified model. It does not prove semantic relationship, but it helps discover associations in a corpus.

In [ ]:
# Select words that occur at least 2 times
selected_words = {word for word, count in word_counts.items() if count >= 2}

edge_weights = Counter()

for tokens in df["tokens"]:
    unique_tokens = sorted(set(token for token in tokens if token in selected_words))
    for w1, w2 in combinations(unique_tokens, 2):
        edge_weights[(w1, w2)] += 1

# Keep stronger edges only
edges = [(w1, w2, weight) for (w1, w2), weight in edge_weights.items() if weight >= 2]

G = nx.Graph()
for w1, w2, weight in edges:
    G.add_edge(w1, w2, weight=weight)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

In [ ]:
plt.figure(figsize=(10, 8))

pos = nx.spring_layout(G, seed=42)
weights = [G[u][v]["weight"] for u, v in G.edges()]

nx.draw_networkx_nodes(G, pos, node_size=500)
nx.draw_networkx_edges(G, pos, width=weights)
nx.draw_networkx_labels(G, pos, font_size=9)

plt.title("Word Co-occurrence Network")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
centrality = nx.degree_centrality(G)
centrality_df = (
    pd.DataFrame(centrality.items(), columns=["word", "degree_centrality"])
    .sort_values("degree_centrality", ascending=False)
)

centrality_df.head(15)

## Interpretation of Network Results

The network highlights terms that connect several parts of the corpus.

Words such as `education`, `planning`, `technology`, `communication`, `trade`, or `production` may function as bridging concepts.  
Their importance does not come only from high frequency, but from their relationship to multiple other terms.

This is a useful DH point: a term can matter because it connects discourses, not merely because it appears often.

# 9. Genre Comparison

Now we compare concept groups by document genre.

This helps distinguish whether modernization vocabulary is distributed across the corpus or concentrated in specific document types.

In [ ]:
concept_by_genre = df.groupby("genre")[list(concept_groups.keys())].sum()
concept_by_genre

In [ ]:
concept_by_genre.plot(kind="bar")
plt.title("Concept Group Frequencies by Genre")
plt.xlabel("Genre")
plt.ylabel("Frequency")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Interpretation of Genre Comparison

The genre comparison shows that:

- economic reports emphasize economy and administration
- cultural reports emphasize culture and education
- technical reports connect technology with administration and planning

The interesting point is not merely that genres differ. That is expected.  
The stronger interpretive point is that administrative vocabulary appears across multiple genres, suggesting that modernization is represented as an institutional process rather than a purely technical or economic one.

# 10. Final Discussion

## Answer to the Research Question

The vocabulary of modernization in this teaching corpus shifts from general development — trade, education, infrastructure — toward a more coordinated language of planning, administration, technology, and institutional control.

The analysis suggests that modernization is represented through four overlapping domains:

1. **economic modernization**: trade, finance, industry, production
2. **cultural modernization**: libraries, education, language, heritage
3. **technical modernization**: radio, transport, machines, electricity
4. **administrative modernization**: planning, coordination, regulation, control

## Humanities Interpretation

A purely quantitative reading would only say that some words occur often.  
A Digital Humanities interpretation asks what these patterns mean.

Here, the computational evidence suggests that modernization discourse is not limited to machines or economic growth. It also includes cultural education, institutional coordination, and administrative control.

## Methodological Reflection

The project combines:

- frequency analysis
- concept-group analysis
- keyword-in-context reading
- co-occurrence network analysis
- genre comparison

This gives a stronger result than any single method alone.

# 11. Limitations

This example has serious limitations:

1. **Synthetic dataset**  
   The documents were created for teaching and are not historical evidence.

2. **Small corpus**  
   Fifteen documents are enough for demonstration, but not enough for robust historical claims.

3. **Simple tokenization**  
   The tokenizer works for basic English but would need adjustment for Latvian or multilingual corpora.

4. **Manual concept groups**  
   The concept categories reflect the researcher's interpretation. This is acceptable, but it must be explicit.

5. **Co-occurrence is approximate**  
   Words appearing in the same document are not necessarily semantically related.

In a real final project, these limitations should be discussed honestly.

# 12. How Students Could Extend This Project

Possible extensions:

- Replace the synthetic corpus with real archival documents
- Add OCR quality analysis
- Compare two time periods
- Use named entity recognition
- Add topic modeling
- Use lemmatization
- Use larger-scale visualizations
- Compare human categories with LLM-generated labels
- Build an interactive dashboard with `ipywidgets` or `streamlit`

# 13. References and Tools

## Python Libraries

- pandas
- matplotlib
- networkx
- regular expressions via Python `re`

## Methodological Concepts

- frequency analysis
- keyword-in-context
- co-occurrence network
- metadata-based comparison
- close and distant reading

## Note for Final Submission

For a real student submission, this section should include:

- dataset citation
- archive or website source
- academic references
- any LLM or AI tool usage disclosure